# The 2026 NeuroGolf Championship

## Score: 5025.75

In [ ]:
from pathlib import Path
import hashlib
import zipfile
import re
import json
from collections import defaultdict, Counter

import numpy as np
import onnx

try:
    import onnxruntime as ort
except Exception:
    ort = None

EXPECTED_FILE_COUNT = 401
EXPECTED_PUBLIC_SCORE = "6335.19"
EXPECTED_MANIFEST_SHA256 = "65e5d817b3195de894b8e1bb97045ed8ae6eeba3b4d79ab8dfcb4c4b88aaa13a"
OUTPUT_ZIP = Path("submission.zip")
MAX_ONNX_BYTES = int(1.44 * 1024 * 1024)
BANNED_OPS = {"LOOP", "SCAN", "NONZERO", "UNIQUE", "SCRIPT", "FUNCTION", "COMPRESS"}

input_root = Path("/kaggle/input")
preferred = input_root / "neurogolf-6335-19-controlled-public-artifact"
search_roots = [preferred] + sorted(input_root.glob("*6335*artifact*")) + sorted(input_root.glob("*neurogolf*artifact*")) + sorted(input_root.glob("*blend*"))
use_full_validation = True


def normalize_task_name(name: str):
    m = re.search(r"task(\d{3})", name.lower())
    if not m:
        return None
    return f"task{m.group(1)}.onnx"


def source_rank(source_name: str):
    s = source_name.lower()
    if "6335" in s:
        return 0
    if "controlled" in s:
        return 1
    if "6285" in s or "6233" in s:
        return 2
    return 9


def manifest_from_items(items):
    lines = []
    for name, data in sorted(items, key=lambda x: x[0]):
        lines.append(f"{name}\t{len(data)}\t{hashlib.sha256(data).hexdigest()}")
    return hashlib.sha256("\n".join(lines).encode()).hexdigest()


def load_zip_models(zip_path: Path):
    out = {}
    with zipfile.ZipFile(zip_path, "r") as zf:
        for name in zf.namelist():
            if not name.lower().endswith(".onnx"):
                continue
            tn = normalize_task_name(Path(name).name)
            if tn is None:
                continue
            out[tn] = zf.read(name)
    return out


def load_dir_models(root: Path):
    out = {}
    for p in root.rglob("task*.onnx"):
        tn = normalize_task_name(p.name)
        if tn is None:
            continue
        out[tn] = p.read_bytes()
    return out


def one_hot_grid(grid):
    t = np.zeros((1, 10, 30, 30), dtype=np.float32)
    for r, row in enumerate(grid):
        if r >= 30:
            break
        for c, v in enumerate(row):
            if c >= 30:
                break
            if 0 <= v < 10:
                t[0, v, r, c] = 1.0
    return t


def is_processable_model(raw: bytes):
    try:
        if len(raw) > MAX_ONNX_BYTES:
            return False
        model = onnx.load_model_from_string(raw)
        onnx.checker.check_model(model, full_check=True)
        for node in model.graph.node:
            if node.op_type.upper() in BANNED_OPS:
                return False
        inferred = onnx.shape_inference.infer_shapes(model, strict_mode=True)
        graph = inferred.graph
        for item in list(graph.input) + list(graph.value_info) + list(graph.output):
            if not item.type.HasField("tensor_type"):
                continue
            tt = item.type.tensor_type
            if not tt.HasField("shape"):
                return False
            for dim in tt.shape.dim:
                if dim.HasField("dim_param"):
                    return False
                if not dim.HasField("dim_value"):
                    return False
        return True
    except Exception:
        return False


def validate_task_model(raw: bytes, task_payload: dict):
    if ort is None:
        return True
    try:
        sess = ort.InferenceSession(raw, providers=["CPUExecutionProvider"])
    except Exception:
        return False
    subsets = ["train", "test", "arc-gen"] if use_full_validation else ["train", "test"]
    for subset in subsets:
        for ex in task_payload.get(subset, []):
            x = one_hot_grid(ex["input"])
            y = sess.run(["output"], {"input": x})[0]
            pred = (y > 0.0).astype(np.float32)
            tgt = one_hot_grid(ex["output"])
            if not np.array_equal(pred, tgt):
                return False
    return True


def source_candidates(roots):
    zip_sources = []
    dir_sources = []
    for root in roots:
        if not root.exists():
            continue
        zips = sorted(root.rglob("submission.zip"))
        for z in zips:
            zip_sources.append(z)
        if not zips:
            dir_sources.append(root)
    return zip_sources, dir_sources


zip_sources, dir_sources = source_candidates(search_roots)

bank = defaultdict(list)
for z in zip_sources:
    try:
        models = load_zip_models(z)
        for k, v in models.items():
            bank[k].append((f"zip:{z.name}", v))
    except Exception:
        pass

for d in dir_sources:
    try:
        models = load_dir_models(d)
        for k, v in models.items():
            bank[k].append((f"dir:{d.name}", v))
    except Exception:
        pass

task_jsons = {}
comp_dir = Path("/kaggle/input/competitions/neurogolf-2026")
if comp_dir.exists():
    for jp in comp_dir.glob("task*.json"):
        tn = normalize_task_name(jp.stem)
        if tn is not None:
            task_jsons[tn] = json.loads(jp.read_text())

selected = {}
usage = Counter()
validation_failures = []
for task_name in sorted(bank.keys()):
    candidates = bank[task_name]
    candidates = sorted(candidates, key=lambda x: (source_rank(x[0]), len(x[1])))
    chosen = None
    processable = []
    for src, raw in candidates:
        if is_processable_model(raw):
            processable.append((src, raw))
    if task_name in task_jsons:
        payload = task_jsons[task_name]
        for src, raw in processable:
            if validate_task_model(raw, payload):
                chosen = (src, raw)
                break
        if chosen is None:
            validation_failures.append(task_name)
    if chosen is None and processable:
        chosen = processable[0]
    if chosen is None and candidates:
        chosen = candidates[0]
    if chosen is not None:
        selected[task_name] = chosen[1]
        usage[chosen[0]] += 1

if selected:
    manifest = manifest_from_items(list(selected.items()))
    if manifest == EXPECTED_MANIFEST_SHA256:
        print("Matched expected 6335.19 manifest")
    else:
        print("Manifest differs from expected 6335.19")
        print(manifest)

with zipfile.ZipFile(OUTPUT_ZIP, "w", zipfile.ZIP_DEFLATED) as zf:
    for name in sorted(selected.keys()):
        zf.writestr(name, selected[name])

zip_sha = hashlib.sha256(OUTPUT_ZIP.read_bytes()).hexdigest()
print(f"Wrote {OUTPUT_ZIP}")
print(f"ONNX files: {len(selected)}")
print(f"zip sha256: {zip_sha}")
print(f"sources used: {dict(usage)}")
if validation_failures:
    print(f"Validation misses: {len(validation_failures)}")
    print(validation_failures[:20])
if len(selected) != EXPECTED_FILE_COUNT:
    print(f"Warning: expected {EXPECTED_FILE_COUNT}, found {len(selected)}")


## Attribution

This release builds on the public NeuroGolf notebook ecosystem and my previous public artifact. Useful public references include:

- `afr1ste/neurogolf-6285-95-public-score-open-solution`
- `artemnazemtsev/neuro-golf-6312`
- `artemnazemtsev/neurogolf-acking-multiple-tasks`
- `artemnazemtsev/neurogolf-blending-is-all-you-need`
- `magmacot/neurogolf-new-blending`
- `jonathanchan/ngc26-constraint-smart-logic-mix-blending`

The point of listing these is practical traceability: public-score artifacts are easier to learn from when the lineage is explicit.
